# Challenge: forward stability

In [ ]:
#    APM41012EP course notebook - Chapter 8 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Challenge: forward stability
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from mpmath import mp
import mpmath
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Explicit Euler method at fixed precision

In [ ]:
#####################################################
class ode_result:
    def __init__(self, y, t):
        self.y = y
        self.t = t

#####################################################
def forward_euler(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y[:,it+1] = yn + dt*np.array(fcn(tn, yn))

    return ode_result(y, t)

#####################################################
def forward_euler_mp(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    
    y = np.empty(int(nt), dtype=object)
    
    y[0] = yini

    tn = tini
    
    for it in range(nt-1):
        yn = y[it]
        y[it+1] = yn + dt*fcn(tn, yn)
        tn = tn + dt

    return y

#####################################################
def forward_euler_mp_bis(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    
    y = np.empty(int(nt), dtype=object)
    
    y[0] = yini

    tn = tini
    
    for it in range(nt-1):
        yn = y[it]
        y[it+1] = yn + dt*fcn(tn, yn)
        tn = tini + (it+1)*dt

    return y

### Results on the Curtiss and Hirschfelder problem

In [ ]:
class curtiss_model:

    def __init__(self, k):
        self.k = k

    def fcn(self, t, y) :
        k = self.k
        cos = np.frompyfunc(mpmath.cos, 1, 1) 
        y_dot = k * (cos(t) - y)
        return y_dot

    def sol(self, yini, t0, t):
        k = self.k
        cos = np.frompyfunc(mpmath.cos, 1, 1) 
        sin = np.frompyfunc(mpmath.sin, 1, 1) 
        exp = np.frompyfunc(mpmath.exp, 1, 1) 
        c0 = (yini - (k/(k*k + 1)) * (k*cos(t0) + sin(t0))) * exp(-k*t0)
        y = (k/(k*k + 1)) * (k*cos(t) + sin(t)) +  c0 * exp(-k*t)
        return y

### Evaluation of the constant 

In [ ]:
yini = 1.
tini = 0.
tend = 1.5
cm = curtiss_model(k=50)
fcn = cm.fcn

nt = np.array([151, 1501, 15001, 150001])

err_inf = np.zeros_like(nt, dtype=float)

for i, nt_i in enumerate(nt):
    sol = forward_euler(tini, tend, nt_i, yini, fcn)
    yerr = np.abs(cm.sol(yini, tini, sol.t) - sol.y[0])
    err_inf[i] = np.linalg.norm(yerr,np.inf) 

dt = (tend-tini)/(nt-1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=dt, y=err_inf, name='linf norm'))
fig.add_trace(go.Scatter(x=dt, y=0.01*dt, mode='lines', line_dash='dot', name='slope 1'))
fig.update_xaxes(type='log', exponentformat='e', title='dt')
fig.update_yaxes(type='log', exponentformat='e', title="Norm of the error")
fig.show()

## Solution at fixed precision

In [ ]:
yini_ref = mp.mpf('1.')
tini_ref = mp.mpf('0.')
tend_ref = mp.mpf('1.5')
k_ref    = mp.mpf('50.')
cm_ref   = curtiss_model(k_ref)

mp01 = mp.clone()
mp01.prec = 20
yini = mp01.mpf('1.') 
tini = mp01.mpf('0.')
tend = mp01.mpf('1.5')
k    = mp01.mpf('50.')
cm   = curtiss_model(k)

print(f"Number of significant digits: {mp01.dps}")

fig = make_subplots(rows=2, cols=1, subplot_titles=("Solution", "Global error"), vertical_spacing=0.1)

nt = np.array([100, 150, 200, 305, 400, 500, 600, 700, 1000, 2000, 5000, 10000, 20000, 80000])

for nt_i in nt: 

    texa = mp.linspace(tini_ref, tend_ref, nt_i)
    texa = np.array(texa)
    yexa = cm_ref.sol(yini_ref, tini_ref, texa)

    ysol = forward_euler_mp(tini, tend, nt_i, yini, cm.fcn)

    err = np.abs(ysol-yexa)

    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(yexa, dtype=float), name='Exact sol.'), row=1, col=1)
    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(ysol, dtype=float), name='Num. sol.'), row=1, col=1)
    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(err, dtype=float), name='error'), row=2, col=1)

fig.data[0].visible = True
fig.data[1].visible = True
fig.data[2].visible = True
    
# create and add slider
steps = []
for i, nt_i in enumerate(nt):
    step = dict(method="update", label = f" {nt_i}",
                args=[{"visible": [el==3*i or el==3*i+1 or el==3*i+2  for el in range(len(fig.data))]}])
    steps.append(step)
        
sliders = [dict(currentvalue={"prefix": "nb points: "}, steps=steps)]

fig.update_yaxes(exponentformat='e', row=2)
fig.update_layout(height=800, sliders=sliders)

Propose an explanation of the behaviour of the error when the time step decreases.

**Time is accumulated in reduced precision**: `tn = tn + dt` is computed with a 20-bit mantissa. When $\Delta t$ becomes small compared to $t_n$, each addition loses a significant fraction of $\Delta t$ (for $nt=80000$, $\Delta t \approx 1.9\cdot10^{-5}$ whereas the ulp of $t_n\approx 1$ is $\approx 2\cdot10^{-6}$). Since the rounding is systematic, $t_n$ drifts: we get $t_N \approx 1.5156$ instead of $1.5$. The scheme therefore integrates the ODE **on a wrong time grid**, while the exact solution is evaluated on the correct grid. Since $|y'| \approx |\sin t| \sim 1$, this produces an error of $\approx 1.5\cdot10^{-2}$: this accounts for most of the observed drift.

## Second solution at fixed precision 

For this test, we replace the update of $t_{n+1} = t_n + \Delta t$ with $t_n = t_0 + n\,\Delta t$ where $\displaystyle n = 1,\dots, \frac{t_f-t_0}{\Delta t}$  

In [ ]:
yini_ref = mp.mpf('1.')
tini_ref = mp.mpf('0.')
tend_ref = mp.mpf('1.5')
k_ref    = mp.mpf('50.')
cm_ref   = curtiss_model(k_ref)

mp01 = mp.clone()
mp01.prec = 20
yini = mp01.mpf('1.') 
tini = mp01.mpf('0.')
tend = mp01.mpf('1.5')
k    = mp01.mpf('50.')
cm   = curtiss_model(k)

print(f"Number of significant digits: {mp01.dps}")

fig = make_subplots(rows=2, cols=1, subplot_titles=("Solution", "Global error"), vertical_spacing=0.1)

nt = np.array([100, 150, 200, 305, 400, 500, 600, 700, 1000, 2000, 5000, 10000, 20000, 80000])

for nt_i in nt: 

    texa = mp.linspace(tini_ref, tend_ref, nt_i)
    texa = np.array(texa)
    yexa = cm_ref.sol(yini_ref, tini_ref, texa)

    ysol = forward_euler_mp_bis(tini, tend, nt_i, yini, cm.fcn)

    err = np.abs(ysol-yexa)

    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(yexa, dtype=float), name='Exact sol.'), row=1, col=1)
    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(ysol, dtype=float), name='Num. sol.'), row=1, col=1)
    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(err, dtype=float), name='error'), row=2, col=1)

fig.data[0].visible = True
fig.data[1].visible = True
fig.data[2].visible = True
    
# create and add slider
steps = []
for i, nt_i in enumerate(nt):
    step = dict(method="update", label = f" {nt_i}",
                args=[{"visible": [el==3*i or el==3*i+1 or el==3*i+2  for el in range(len(fig.data))]}])
    steps.append(step)
        
sliders = [dict(currentvalue={"prefix": "nb points: "}, steps=steps)]

fig.update_yaxes(exponentformat='e', row=2)
fig.update_layout(height=800, sliders=sliders)

Why is the behaviour of the error when the time step decreases not the same as for the previous solution?

We are working on a one-step scheme; stability with respect to perturbations and consistency, and even the order $p$, lead us to an estimate of the global error:

$$
\max_{0\le n\le N} \vert e_n \vert \le M (  \vert e_0 \vert + C T \Delta t^p).
$$

However, in this type of approach, we have not taken into account machine arithmetic and the round-off errors in the operations. In this framework, we can introduce into the one-step scheme the perturbations associated with the operations that are performed. 

Let us assume that the scheme is explicit, for the sake of simplicity, and let us introduce the round-off errors:

$$
\hat y_{n+1} = \hat  y_n + \Delta t_n \left( \phi(t_n,\hat y_n;\Delta t_n) + \epsilon^{\phi}_n \right) + \epsilon^{c}_n,
$$
where the quantities $\epsilon^{\phi}_n$ (round-off error in the computation of $\phi$) and $\epsilon^{c}_n$ (round-off error in the computation of $\hat y_{n+1}$) are bounded by the machine error $\varepsilon_M$.

A direct application of stability with respect to perturbations leads us to the following estimate:
$$\max_{0\le n\le N} \vert e_n \vert \le M \left(  \vert e_0 \vert + C T \Delta t^p + T \epsilon^{\phi} + \frac{T\epsilon^{c}}{\Delta t}\right),
$$
where $\epsilon^{\phi}$ and $\epsilon^{c}$ are upper bounds of the perturbations that have been introduced, and are therefore of the order of the machine error.

A somewhat unexpected phenomenon then appears, and one that is somewhat contrary to the intuition that theoretical numerical analysis has led us to have, with an asymptotic vision of the time step tending to zero. **If the time step is too small, the large number of operations with rounding can generate an accumulation of errors and degrade the global error**. 

There exists a kind of optimal time step for which the round-off errors associated with the multiplication of small time steps are of the same order of magnitude as the local truncation errors obtained by theoretical numerical analysis:
$$
\Delta t_{\mathrm{opt}} = \left( \frac{\epsilon^{c}}{C} \right)^{\frac{1}{p+1}} \simeq \left( \frac{\epsilon_{M}}{C} \right)^{\frac{1}{p+1}}.
$$

In our example:
- $\epsilon_{M} = 2^{1-p}$ the machine epsilon (gap between $1$ and the next floating-point number), with $p = 20$ the number of significant bits of the floating-point numbers used
- $C = 0.01$ (cf. the cell displaying the error as a function of the time step of the scheme)
- $p = 1$ (order of the explicit Euler method)

We then obtain:

In [ ]:
eps = np.power(2.,-mp01.prec+1)
C = 0.01
p = 1

print("eps = ", eps)
print("C = ", C)
print("p = ", p)

dt_opt = np.power(eps/C, 1/(p+1))
print("dt_opt = ", dt_opt)

print("\noptimal time step observed numerically (exact time) for nt ≈ 5000, i.e.:")
print("dt_opt num. ≈", (tend-tini)/(5000-1))

On the graph, the error keeps decreasing until $nt \approx 5000$: no minimum appears at $\Delta t_{\mathrm{opt}} = (\varepsilon_M/C)^{1/2}$.

With the time computed exactly ($t_n = t_0 + n\Delta t$), nothing special happens around $\Delta t \approx 1.4\cdot10^{-2}$ ($nt \approx 110$). For $nt = 100$ and $nt = 150$, the observed errors ($1.5\cdot10^{-4}$ and $1.0\cdot10^{-4}$) are exactly the truncation error $C\,\Delta t$.

The reason is that $(\varepsilon_M/C)^{1/2}$ is the point where the truncation error $C\,\Delta t$ equals an **upper bound** of the rounding error, $T\varepsilon_M/\Delta t$, and not the rounding error itself.

The problem is stiff and dissipative ($k = 50$): a perturbation introduced at step $n$ is damped by a factor $(1-k\Delta t)$ at each step. Rounding errors therefore do not add up over the whole interval $[0,T]$, but only over a characteristic time $1/k$. The term $T\epsilon^c/\Delta t$ is replaced by $\epsilon^c/(k\Delta t)$, which gives

$$
\Delta t_{\mathrm{opt}} \simeq \left(\frac{\varepsilon_M}{k\,C}\right)^{1/2} \approx 2.0\cdot10^{-3},
$$

which corresponds to $nt \approx 770$, a factor of about 6.5 away from what is observed numerically.

This is again an upper bound, because it assumes the worst case where **all rounding errors have the same sign**.

## Kahan compensated summation

In $\hat y_{n+1} = \hat y_n + \delta_n$ with $\delta_n = \Delta t\,\phi(t_n,\hat y_n)$, the increment $\delta_n$ is small compared to $\hat y_n$: the addition loses the low-order bits of $\delta_n$. This is exactly the perturbation $\epsilon^c_n$, which is responsible for the $T\epsilon^c/\Delta t$ term.

Kahan summation stores in a compensation variable $c$ the part of the increment lost to rounding, and reinjects it at the next step:

```
z = delta - c
s = y + z
c = (s - y) - z     # what was lost in y + z
y = s
```

The accumulated rounding error of the sum becomes $O(\varepsilon_M) + O(N\varepsilon_M^2)$ instead of $O(N\varepsilon_M)$: **the $\epsilon^c/\Delta t$ term disappears**. What remains is:

- the $T\epsilon^\phi$ term (rounding errors in the evaluation of $\phi$), which does not grow as $\Delta t \to 0$ since it is multiplied by $\Delta t$;
- a floor of the order of $\varepsilon_M$, due to the storage of $\hat y_n$ itself (and to the fact that $\phi$ is evaluated at $\hat y_n$ rather than at $\hat y_n + c$).

In [ ]:
#####################################################
def forward_euler_mp_kahan(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    
    y = np.empty(int(nt), dtype=object)
    
    y[0] = yini

    c = 0*yini   # Kahan compensation

    tn = tini
    
    for it in range(nt-1):
        yn = y[it]
        z = dt*fcn(tn, yn) - c
        s = yn + z
        c = (s - yn) - z   # what was lost in yn + z
        y[it+1] = s
        tn = tini + (it+1)*dt

    return y

In [ ]:
yini_ref = mp.mpf('1.')
tini_ref = mp.mpf('0.')
tend_ref = mp.mpf('1.5')
k_ref    = mp.mpf('50.')
cm_ref   = curtiss_model(k_ref)

mp01 = mp.clone()
mp01.prec = 20
yini = mp01.mpf('1.') 
tini = mp01.mpf('0.')
tend = mp01.mpf('1.5')
k    = mp01.mpf('50.')
cm   = curtiss_model(k)

print(f"Number of significant digits: {mp01.dps}")

fig = make_subplots(rows=2, cols=1, subplot_titles=("Solution", "Global error"), vertical_spacing=0.1)

nt = np.array([100, 150, 200, 305, 400, 500, 600, 700, 1000, 2000, 5000, 10000, 20000, 80000])

for nt_i in nt: 

    texa = mp.linspace(tini_ref, tend_ref, nt_i)
    texa = np.array(texa)
    yexa = cm_ref.sol(yini_ref, tini_ref, texa)

    ysol = forward_euler_mp_kahan(tini, tend, nt_i, yini, cm.fcn)

    err = np.abs(ysol-yexa)

    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(yexa, dtype=float), name='Exact sol.'), row=1, col=1)
    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(ysol, dtype=float), name='Num. sol.'), row=1, col=1)
    fig.add_trace(go.Scattergl(visible=False, x=np.array(texa, dtype=float), y=np.array(err, dtype=float), name='error'), row=2, col=1)

fig.data[0].visible = True
fig.data[1].visible = True
fig.data[2].visible = True
    
# create and add slider
steps = []
for i, nt_i in enumerate(nt):
    step = dict(method="update", label = f" {nt_i}",
                args=[{"visible": [el==3*i or el==3*i+1 or el==3*i+2  for el in range(len(fig.data))]}])
    steps.append(step)
        
sliders = [dict(currentvalue={"prefix": "nb points: "}, steps=steps)]

fig.update_yaxes(exponentformat='e', row=2)
fig.update_layout(height=800, sliders=sliders)

The global error then behaves like $C\,\Delta t + O(\varepsilon_M)$ and decreases until it saturates at machine precision, without rising again.

## Summary

Global error in discrete $L^2$ norm, $\displaystyle \Vert e \Vert_{2} = \Big( \Delta t \sum_{n} \vert e_n \vert^2 \Big)^{1/2}$, as a function of the time step for the three implementations, all in 20-bit arithmetic:

- **accumulated time**: $t_{n+1} = t_n + \Delta t$ (`forward_euler_mp`);
- **exact time**: $t_n = t_0 + n\Delta t$ (`forward_euler_mp_bis`);
- **exact time + Kahan**: exact time and Kahan compensated summation on $\hat y_n$ (`forward_euler_mp_kahan`).

In [ ]:
yini_ref = mp.mpf('1.')
tini_ref = mp.mpf('0.')
tend_ref = mp.mpf('1.5')
k_ref    = mp.mpf('50.')
cm_ref   = curtiss_model(k_ref)

mp01 = mp.clone()
mp01.prec = 20
yini = mp01.mpf('1.') 
tini = mp01.mpf('0.')
tend = mp01.mpf('1.5')
k    = mp01.mpf('50.')
cm   = curtiss_model(k)

schemes = {'accumulated time': forward_euler_mp,
           'exact time':       forward_euler_mp_bis,
           'exact time + Kahan': forward_euler_mp_kahan}

nt = np.array([100, 150, 200, 305, 400, 500, 600, 700, 1000, 2000, 3000, 5000, 10000, 20000, 40000, 80000])
dt = (float(tend)-float(tini)) / (nt-1)

err_l2 = {name: np.zeros(nt.size) for name in schemes}

for i, nt_i in enumerate(nt):

    texa = mp.linspace(tini_ref, tend_ref, nt_i)
    texa = np.array(texa)
    yexa = cm_ref.sol(yini_ref, tini_ref, texa)

    for name, scheme in schemes.items():
        ysol = scheme(tini, tend, nt_i, yini, cm.fcn)
        err = np.array(np.abs(ysol-yexa), dtype=float)
        err_l2[name][i] = np.sqrt(dt[i]*np.sum(err**2))   # discrete L2 norm on [tini, tend]

u = 2.**(-mp01.prec)   # unit roundoff

fig = go.Figure()
for name in schemes:
    fig.add_trace(go.Scatter(x=dt, y=err_l2[name], mode='lines+markers', name=name))
fig.add_trace(go.Scatter(x=dt, y=0.01*dt, mode='lines', line_dash='dot', name='slope 1'))
fig.add_trace(go.Scatter(x=dt, y=u*np.ones_like(dt), mode='lines', line_dash='dash', name='unit roundoff 2^-20'))
fig.update_xaxes(type='log', exponentformat='e', title='dt')
fig.update_yaxes(type='log', exponentformat='e', title='Global error (L2 norm)')
fig.update_layout(height=550, legend=dict(x=0.02, y=0.98, bgcolor='rgba(0,0,0,0)'))
fig.show()